# Part 0 : Setup & Imports

This notebook accompanies the written report for the Applied Investments portfolio strategy project (Prof. Amit Goyal, HEC Lausanne, Fall 2026) and documents every step needed to reproduce the results. The research question is: **"Can a momentum strategy we design keep the momentum premium, reduce its crashes, and still earn alpha after costs, out of sample?"**

Momentum (buy the past 12-month winners, sell the losers) is one of the most robust anomalies in the literature, and one of the few that the Fama-French five-factor model cannot explain. It suffers from two problems that make it hard to defend in front of an investment committee: violent, sudden crashes (spring 2009) and high turnover that erodes the premium after trading costs. Our strategy keeps the premium while addressing both, and is built as a cascade so the contribution of each step can be measured separately.

We test three hypotheses, each tied to a row of the cascade table:

- **H1 (signal)** : the momentum window selected on 1963–2011 does at least as well as the standard 12-1 after 2012.
- **H2 (our rule)** : scaling down the short leg in bear markets with high loser volatility reduces maximum drawdown and the worst month without lowering the Sharpe ratio, and the gain is not explained by time-varying market exposure.
- **H3 (implementation)** : after the turnover buffer and transaction costs, the final strategy has a positive alpha against FF5 + UMD.

All design choices are fixed on the in-sample period (1963–2011). The period 2012–today is used only as an out-of-sample test, and no parameter changes after it has been looked at.

Course rules that shape the design: US equity data only, a strategy evaluated with alphas against standard asset pricing models, a report of at most 20 pages (excluding tables, graphs and references), and a 20-minute presentation to the investment committee of our asset management firm.

> **Committee message** : "The momentum premium, without its crashes, implementable after costs, and tested on data we never used to design it."

## 0.1 : Global Conventions

Definitions agreed on by the group before any code is run, so the report, the slides and the code all use the same constants:

- Rebalancing: monthly, on the last trading day of month $t$ (return-based signals change often, so monthly rebalancing is the natural choice). Holding period returns start at $t+1$.
- Value weights: the return of month $t+1$ is weighted by market capitalisation at the end of month $t$.
- Risk-free rate: one-month T-bill from the Kenneth French Data Library.
- Annualisation: mean × 12, volatility × $\sqrt{12}$ for monthly series; × 252 / $\sqrt{252}$ for daily series.
- Sample: start and end dates, in-sample / out-of-sample cut-off (31 December 2011).
- Missing data stay missing: a missing return is never filled with zero.
- Delisting returns: included in the month of delisting.
- Transaction costs: spread plus price impact model from lecture 3 (Part 4), with an assumed fund size; fixed 10 / 25 / 50 bp as sensitivity.

---
# Part 1 : Data Loading & Cleaning

## 1.1 : Load the Stock-Level Database

We load US stock data from CRSP via WRDS (to be confirmed by the group). Following lecture 1, the sample includes all stocks, alive and dead, and should be as long as possible: at least 50 years, covering the oil crisis, the internet bubble, the global financial crisis and COVID.

Required fields: adjusted price and return (daily and monthly), PERMNO, ticker, shares outstanding, volume, share code and exchange code, delisting return. Bid and ask quotes, or daily high and low prices, are needed for the spread estimates in Part 4.

This is the main data source for the whole notebook.

## 1.2 : Data Audit

Four checks must pass before any strategy code is written, since each one can invalidate the backtest:

- **Survivorship bias** : the database must include delisted firms (bankruptcies, takeovers) and their delisting returns. Without them, losers that went bankrupt disappear and momentum is artificially inflated.
- **Adjusted prices** : returns must account for splits and dividends (total return). Otherwise a 2:1 split appears as a 50% loss.
- **Recycled tickers** : a ticker can successively refer to several firms, so firms are tracked with PERMNO, and abnormal price jumps are flagged.
- **Look-ahead bias** : every piece of information used at date $t$ must be known at $t$.

We report the number of firms, the number of delisted firms, and any anomalies found.

## 1.3 : Daily & Monthly Returns

Daily returns are compounded into monthly returns:

$$r_{i,t} = \prod_{d \in t}(1 + r_{i,d}) - 1$$

Delisting returns are added in the month of delisting. Daily returns are kept for the realised volatility measures in Parts 6 and 8.

## 1.4 : External Data

All external series are free and downloaded once:

| Data | Source | Use |
|------|--------|-----|
| Fama-French 3 and 5 factors, risk-free rate (monthly and daily) | Kenneth French Data Library | Excess returns, residual momentum, alphas |
| Momentum factor UMD | Kenneth French Data Library | Validation of WML, Carhart and FF5 + UMD alphas |
| NYSE market cap breakpoints (ME) | Kenneth French Data Library | Micro-cap exclusion |
| q-factors (Hou-Xue-Zhang) | global-q.org | q-factor alphas |
| NBER recession dates | nber.org | Sub-period analysis |

## 1.5 : Investment Universe

At the end of each month $t$, a stock is in the universe if:

- it is a US common stock (share codes 10 and 11; excluding ETFs, funds, REITs and ADRs) listed on NYSE, AMEX or NASDAQ;
- its price is above \$5 at the formation date;
- its market capitalisation is above the 20th NYSE percentile (Hou, Xue & Zhang 2020). Lecture 3 shows that micro-caps account for most of momentum's trading costs, which is why we exclude them;
- it has at least 8 monthly returns over the formation window (and 24 out of 36 for residual momentum).

Portfolios are value-weighted in the main specification, as preferred in lecture 1. Equal-weighted deciles are reported alongside value-weighted ones for the baseline (Part 3.3), but the strategy itself is evaluated value-weighted. Novy-Marx & Velikov (2016) show equal weighting overstates performance by overweighting small, expensive-to-trade stocks.

## 1.6 : Sample Split

| Period | Role | What we are allowed to do |
|--------|------|---------------------------|
| 1963–2011 | Design (in-sample) | Choose the signal, set thresholds and the buffer, estimate everything |
| 2012–today | Test (out-of-sample) | Only report results; no parameter changes after looking |

2012 is the publication year of Novy-Marx's "Is momentum really momentum?", so the test period is also a clean test of the echo debate. Because the signal needs 12 months and residual momentum 36 months of history, the first holding period starts about three years after the data begin.

We define the cut-off date here once as a constant so every later section uses the same value.

**Power limitation** : 13–14 years of monthly data give a standard error of about 0.27 on an annual Sharpe ratio. Out-of-sample results are therefore reported descriptively, with confidence intervals.

---
# Part 2 : Descriptive Statistics

## 2.1 : Universe Over Time

We plot the number of stocks in the universe each month and the share of total US market capitalisation it covers. This shows how much the size and price filters remove and whether the universe is stable over time.

## 2.2 : Summary Statistics of Factors

Annualised mean, volatility, Sharpe ratio, skewness and excess kurtosis of MKT, SMB, HML, RMW, CMA and UMD, for the full sample, in-sample and out-of-sample. UMD's strongly negative skewness is the first sign of the crash problem studied in Part 3.

_Interpretation: to be written once results are in._

---
# Part 3 : Baseline Momentum (Row A)

We follow the portfolio analysis steps from lecture 1: define the sort variable, compute breakpoints, group stocks into portfolios, average returns, compute portfolio characteristics, and summarise.

## 3.1 : Standard 12-1 Signal

At the end of each month $t$, we compute each stock's cumulative return from $t-12$ to $t-2$:

$$MOM_{i,t} = \prod_{s=t-12}^{t-2}(1 + r_{i,s}) - 1$$

The most recent month is skipped because one-month returns show short-term reversal, which would contaminate the signal.

## 3.2 : NYSE Decile Breakpoints & Sorts

We use 10 portfolios, with breakpoints computed from NYSE stocks only and then applied to the whole universe. As in lecture 1, NYSE breakpoints avoid sorts dominated by tiny but numerous AMEX and NASDAQ stocks. This is also how UMD is built, which makes the validation in Part 3.5 meaningful.

## 3.3 : Decile Portfolios & WML

We form equal- and value-weighted decile portfolios, held for one month and then rebalanced. The baseline strategy goes long decile 10 (winners) and short decile 1 (losers):

$$WML_{t+1} = r^{W}_{t+1} - r^{L}_{t+1}$$

We report the average return and standard deviation of all ten deciles and of 10−1, plus the cumulative growth of each decile, to check that the relationship is roughly monotonic.

## 3.4 : Decile Characteristics

For each decile: average number of stocks, average market capitalisation, and average signal value (sanity check that the sort works). This shows where in the size distribution the winners and losers sit, which matters for trading costs.

## 3.5 : Validation Against UMD

Key quality check: value-weighted WML should have a correlation of about 0.9 with Kenneth French's UMD (not 1, since UMD is built from 2×3 size-momentum sorts) and a similar mean. If not, there is a problem in the data or the construction, and we do not move on.

_Interpretation: to be written once results are in._

## 3.6 : Momentum Crashes in Our Data

Before any correction, we document the problem in our own sample: the ten worst months of WML, its skewness, and the returns of the long and short legs from March to May 2009. The point is to show that crashes come from the short leg: after a market decline the losers are high-beta stocks that rebound sharply when the market recovers.

This analysis motivates the short-leg rule in Part 6 and makes the strategy's logic our own rather than a replication.

_Interpretation: to be written once results are in._

---
# Part 4 : Turnover & Transaction Cost Framework

The same cost model is applied to every row of every table, so no version of the strategy is favoured. It is defined here because the signal selection in Part 5 already uses net returns. The approach follows lecture 3.

## 4.1 : Portfolio Turnover

Between rebalancing dates, weights drift with price returns: $w^{*}_{i,t+1} = w_{i,t}\,(1 + Rx_{i,t+1})$. The trade needed to reach the new target weights is:

$$\Delta w_{i,t+1} = w_{i,t+1} - \frac{w^{*}_{i,t+1}}{\sum_j w^{*}_{j,t+1}}$$

$$\text{Turn}_{t+1} = \sum_i \left|\Delta w_{i,t+1}\right|$$

This includes stocks that stay, exit and enter. A full change of composition is 200% for one leg, so the maximum for a long-short portfolio is 400%. For scaled versions (our rule, Moreira-Muir), turnover also includes trades caused by changes in $k_t$ or in leverage, not only changes in composition.

## 4.2 : Cost Model: Spread + Price Impact

Each trade pays half the bid-ask spread plus a price impact:

$$\text{Cost}_{t} = \sum_i \left|\Delta w_{i,t}\right| \left(\frac{s_{i,t}}{2} + \lambda_i \frac{\text{Trade size}_{i,t}}{\text{ADV}_{i,t}}\right)$$

- **Spread $s_{i,t}$** : from CRSP closing bid and ask quotes where available, otherwise the Corwin & Schultz (2012) high-low estimator. As in lecture 3, remaining missing values are replaced with the mean spread of the stock's size category.
- **Price impact** : $\lambda$ = 2.5, 1.5 and 1.0 for micro, small and large stocks, i.e. 25, 15 and 10 bp for a trade of 1% of average daily volume (ADV), as in lecture 3.
- **Trade size** depends on an assumed fund size (base case to be fixed in Part 9, e.g. \$100 million), with a larger fund as sensitivity.

## 4.3 : Fixed-Cost Scenarios

As a simple, model-free cross-check, we also apply fixed costs of 10, 25 and 50 bp per unit of turnover to every row. These are reported in the robustness section (Part 14.1).

## 4.4 : Sanity Check Against Lecture 3

Our baseline numbers should be in the same range as the momentum results shown in lecture 3 for non-micro-cap, value-weighted stocks: about 149% monthly turnover for 10−1, and about 82% under the 10%/20% rule. Large deviations point to an error in the turnover calculation.

## 4.5 : Short-Selling Limitation

Stock borrowing fees are not modelled. This is stated as a limitation, and the short leg's contribution to returns is reported separately in Part 10.5.

---
# Part 5 : Signal Selection (In-Sample, Row B)

## 5.1 : Candidate Signals

Four candidate signals, all using the same NYSE decile sort and long-short construction as Part 3:

| Signal | Formation window | Source |
|--------|------------------|--------|
| Standard 12-1 | $t-12$ to $t-2$ | Jegadeesh & Titman (1993) |
| Intermediate 12-7 | $t-12$ to $t-7$ | Novy-Marx (2012) |
| Recent 6-2 | $t-6$ to $t-2$ | Novy-Marx (2012); Goyal & Wahal (2015) |
| Residual 12-1 | FF3 residuals $t-12$ to $t-2$, scaled by their volatility | Blitz, Huij & Martens (2011) |

## 5.2 : Residual Momentum

For each stock, we estimate over the previous 36 months:

$$r_{i,s} - r_{f,s} = \alpha_i + \beta_i\, MKT_s + s_i\, SMB_s + h_i\, HML_s + \varepsilon_{i,s}$$

The signal is the sum of residuals from $t-12$ to $t-2$, divided by their standard deviation:

$$RMOM_{i,t} = \frac{\sum_{s=t-12}^{t-2} \hat{\varepsilon}_{i,s}}{\sigma(\hat{\varepsilon}_i)}$$

At least 24 of the 36 monthly returns must be available.

## 5.3 : In-Sample Portfolio Comparison

For each of the four signals, on 1963–2011 only: annualised return, volatility, gross and net Sharpe ratio, maximum drawdown, skewness and average turnover.

## 5.4 : Fama-MacBeth Horse Race

Portfolio sorts look at one signal at a time. Fama-MacBeth regressions (lecture 1) let us control for other characteristics and test the signals against each other, which is the format Novy-Marx (2012) and Goyal & Wahal (2015) use for the echo debate. Each month we run the cross-sectional regression:

$$R_{i,t+1} = \gamma_{0,t} + \gamma_{1,t}\,X_{i,t} + \gamma_{2,t}\log(ME_{i,t}) + \gamma_{3,t}\,R_{i,t} + e_{i,t+1}$$

where $X_{i,t}$ contains the candidate signals and $R_{i,t}$ is the previous month's return (short-term reversal). Book-to-market is added as a control if accounting data are available. We report the time-series averages $\bar{\gamma} = \frac{1}{T}\sum_t \gamma_t$ with their t-statistics, on 1963–2011 only.

This is supporting evidence. The selection itself follows the pre-registered rule in 5.5.

## 5.5 : Pre-Registered Selection Rule

The rule is fixed before any result is seen: we keep the signal with the highest in-sample net Sharpe ratio (cost model of Part 4.2). If the best signal beats the standard 12-1 by less than 0.05, we keep the 12-1 as the simplest choice.

The selected signal defines row B. The full comparison of all four signals after 2012 is reported in Part 10.2 regardless of the winner. We only test the US part of the echo debate.

_Interpretation: to be written once results are in._

---
# Part 6 : Short-Leg Crash Protection (Our Rule, Row C)

This is our own contribution. Daniel & Moskowitz (2016) scale the whole WML portfolio using forecasts of its mean and variance, and Cooper, Gutierrez & Hameed (2004) condition momentum on the market state. Our rule differs: it acts on the short leg only, uses two observable conditions with no forecasting model, leaves the long leg untouched, and is tested out of sample.

## 6.1 : Bear Market Indicator

$\text{Bear}_t = 1$ if the cumulative excess market return over the past 24 months (up to $t$) is negative. The 24-month window follows Daniel & Moskowitz (2016).

## 6.2 : Loser Volatility Indicator

$\text{HighVol}_t = 1$ if the realised volatility of the loser portfolio's daily returns over month $t$ is above its median, where the median is computed on an expanding window using data up to $t$ only (no look-ahead).

## 6.3 : Short-Leg Scaling

When both conditions hold, the short leg is reduced:

$$k_t = 1 - \delta \cdot \mathbf{1}[\text{Bear}_t] \cdot \mathbf{1}[\text{HighVol}_t]$$

$$r^{\text{final}}_{t+1} = \left(r^{W}_{t+1} - r_{f,t+1}\right) - k_t\left(r^{L}_{t+1} - r_{f,t+1}\right)$$

$\delta = 0.5$ is fixed a priori, not optimised.

Consequence to report: when $k < 1$ the portfolio is net long the market, precisely in bear markets. This helps during a rebound (2009) and hurts if the market keeps falling (late 2008). We address this with the conditional-beta test in Part 11.3 and a beta-hedged variant in the appendix (Part 14.3).

## 6.4 : Trigger Count & Episodes

Number of months the rule is active, in-sample and out-of-sample, and the list of episodes (dates). If the rule rarely or never triggers after 2012, the report says so clearly.

## 6.5 : Sensitivity to δ (In-Sample Only)

Results for $\delta \in \{0.25, 0.5, 0.75, 1\}$ on 1963–2011, shown as sensitivity only. The base value stays at 0.5 whatever the results.

## 6.6 : Does the Trigger Predict Crashes? (Predictive Regression & OOS $R^2$)

Our rule is a timing signal, so we evaluate it the way lecture 4 evaluates market predictors. The predictive regression is:

$$r^{L}_{t+1} - r_{f,t+1} = a + b\,\mathbf{1}[\text{Bear}_t]\,\mathbf{1}[\text{HighVol}_t] + e_{t+1}$$

A positive $b$ means losers rebound more after a trigger, which is the risk our rule avoids. In-sample, we report $b$, its t-statistic and $R^2$. Out of sample, we run the lecture 4 horse race: investor A re-estimates the regression each month on an expanding window, while investor B uses the historical mean. We report

$$R^2_{OOS} = 1 - \frac{\sum_t (E^{A}_t)^2}{\sum_t (E^{B}_t)^2}$$

A positive $R^2_{OOS}$ means the trigger beats the naive benchmark in real time.

_Interpretation: to be written once results are in._

---
# Part 7 : Turnover Buffer (Row D, Final Strategy)

## 7.1 : Buy/Hold Spread Rule

This is the 10%/20% cost-mitigation rule from lecture 3 (Novy-Marx & Velikov 2016). A stock enters the long leg when it reaches decile 10 and stays as long as it is in deciles 9–10. Symmetrically, it enters the short leg at decile 1 and stays while in deciles 1–2. It works because stocks just outside the top decile have almost the same expected return as those just inside, so holding them is a cheap substitute for trading. Buffer widths are fixed a priori.

## 7.2 : Turnover Reduction

We compare average monthly turnover, spread costs, price impact and net returns of row C (no buffer) and row D (buffer). The signal is almost unchanged but the number of trades should fall sharply, close to halving as in lecture 3.

_Interpretation: to be written once results are in._

---
# Part 8 : Competitor: Real-Time Moreira-Muir Volatility Scaling (Row E)

Moreira & Muir (2017) is kept as a competitor that our rule must beat, not as a layer. Row E = row B + real-time volatility scaling + turnover buffer, so it is directly comparable with row D.

## 8.1 : Realised Variance

Realised variance of the long-short portfolio over month $t$, from its daily returns:

$$\sigma^2_t = \sum_{d \in t}\left(r_d - \bar{r}_t\right)^2$$

## 8.2 : Scaled Returns with Real-Time $c_t$

$$r^{*}_{t+1} = \frac{c_t}{\sigma^2_t}\, r_{t+1}$$

$c_t$ is estimated on an expanding window using data up to $t$ only, so the managed strategy targets the same volatility as the unmanaged one without using future information. Leverage $c_t/\sigma^2_t$ is capped at 2.

The full-sample $c$ version goes in the appendix, flagged as having look-ahead bias. The report cites the finding that volatility-management gains largely disappear in real time and after costs (Cederburg, O'Doherty, Wang & Yan 2020; Barroso & Detzel 2021).

## 8.3 : Turnover Including Leverage Changes

Turnover for row E includes the trades caused by month-to-month changes in leverage, using the same cost model as Part 4.

---
# Part 9 : Design Freeze

Before any out-of-sample number is computed, all design choices are recorded here and committed to the code repository:

| Parameter | Value |
|-----------|-------|
| Selected signal | _from Part 5.5_ |
| Bear indicator window | 24 months |
| Loser volatility threshold | Expanding median |
| δ | 0.5 |
| Buffer | Enter 10 / hold 9–10; enter 1 / hold 1–2 |
| Moreira-Muir leverage cap | 2 |
| Cost model | Half-spread + price impact (λ = 2.5 / 1.5 / 1.0) |
| Assumed fund size | _to be fixed_ |

From this point on, no parameter is changed. The out-of-sample panel in Part 10 is computed with exactly these settings.

---
# Part 10 : Main Results: Cascade Table

## 10.1 : Cascade Table

One panel for 1963–2011 (in-sample) and one for 2012–today (out-of-sample), with the same rows and columns:

| Row | Version |
|-----|---------|
| A | Standard 12-1 momentum (benchmark) |
| B | A with the selected signal (Part 5) |
| C | B + our short-leg rule (Part 6) |
| D | C + turnover buffer (Part 7): **final strategy** |
| E | B + real-time Moreira-Muir + buffer (competitor, Part 8) |

Columns: annualised return and volatility, Sharpe ratio (with bootstrap confidence interval from Part 12.4), maximum drawdown, skewness, worst month, average monthly turnover, market beta, and net return and Sharpe after costs. The D vs E comparison puts our rule head-to-head with the literature's.

_Interpretation: to be written once results are in._

## 10.2 : Out-of-Sample Signal Comparison

All four candidate signals after 2012, reported in full regardless of which one was selected. This is our out-of-sample answer to the echo debate (H1).

_Interpretation: to be written once results are in._

## 10.3 : Cumulative Returns & Drawdowns

Cumulative log returns and drawdown paths of rows A to E over the full sample, with the in-sample / out-of-sample cut-off and NBER recessions shaded.

## 10.4 : 2009 Case Study

Monthly returns of each version from March to May 2009, the largest momentum crash in the sample, split into long-leg and short-leg contributions. This is the most direct visual test of whether our rule works when it matters.

_Interpretation: to be written once results are in._

## 10.5 : Long-Leg vs Short-Leg Contribution

For rows A to E, we split the return into the long leg (winners minus risk-free) and the short leg (risk-free minus losers), in-sample and out-of-sample, with each leg's Sharpe ratio and share of total return. Since borrowing fees are not modelled, this shows how much of the result depends on the short side, and how much would remain for a long-only or short-constrained investor.

_Interpretation: to be written once results are in._

---
# Part 11 : Alphas & Market Timing Test

## 11.1 : Factor Model Alphas

Following lecture 2, we run time-series regressions of the excess return of each row on the factors:

$$r_t = \alpha + \beta' f_t + \varepsilon_t$$

Models: CAPM, Fama-French 3, Carhart (FF3 + UMD), FF5 + UMD and the Hou-Xue-Zhang q-factor model. FF5 + UMD is the main model because it controls for momentum itself. The q-factor model is a demanding test, since lecture 2 shows it can largely explain momentum through its profitability factor.

t-statistics use Newey-West standard errors with 6 lags. Following Harvey, Liu & Zhu (2016), significance is judged against $t > 3$ rather than 1.96 to account for data mining. The key test is the alpha of row D against FF5 + UMD (H3).

## 11.2 : Incremental Value

To isolate the contribution of each step, we regress C on B and D on C. A significant intercept means the step adds something the previous row does not already capture.

## 11.3 : Conditional Beta Test

Since our rule changes market exposure in bear markets, a static CAPM alpha is not enough. We estimate:

$$r_t = \alpha + \beta\, MKT_t + \beta_B\, \mathbf{1}[\text{Bear}_{t-1}]\, MKT_t + \varepsilon_t$$

and check whether the rule's gain survives when beta is allowed to differ in bear months (H2).

_Interpretation: to be written once results are in._

---
# Part 12 : Placebo & Bootstrap Tests

Shuffling the order of a strategy's monthly returns changes neither its mean, volatility nor Sharpe ratio, so on its own it tests nothing. We use four versions that do work.

## 12.1 : Timing Placebo (Tests Our Rule)

Keep the same number of triggered months but place them at random dates, then recompute row C. Repeat 1,000 times. If the true timing beats random timing, the rule adds value.

## 12.2 : Cross-Sectional Placebo (Tests the Signal)

Each month, randomly reassign the signal across stocks and rebuild the long-short portfolio. Repeat 1,000 times and compare the distribution with the true strategy.

## 12.3 : Time-Shuffle Test

Shuffle the months to destroy volatility clustering, then recompute rows C and E. If each rule's benefit disappears, it genuinely comes from timing.

## 12.4 : Block Bootstrap Confidence Intervals

Block bootstrap with 12-month blocks to obtain confidence intervals for the Sharpe ratio and maximum drawdown of rows A to E. These intervals feed the cascade table in Part 10.1.

_Interpretation: to be written once results are in._

---
# Part 13 : Portfolio Implications

The investment committee does not hold the strategy on its own: the question is what it adds to the portfolio the firm already holds. This Part translates the results into those terms.

## 13.1 : Correlation with Existing Exposures

Correlation of row D with the market, value (HML), the other Fama-French factors and UMD, in-sample and out-of-sample. We also report the strategy's worst calendar year next to the market's worst year.

## 13.2 : Adding the Strategy to a Market Portfolio

We combine the market portfolio with rows A and D and compare the Sharpe ratio, volatility and maximum drawdown of the combined portfolio. We also report the weight on the strategy that maximises the in-sample Sharpe ratio, and apply that same weight out of sample. The question is whether row D improves the existing portfolio more than standard momentum does.

## 13.3 : Capacity

Average market capitalisation and average dollar volume of the stocks held in each leg, and the share of ADV our trades would represent at different fund sizes. We report the fund size at which price impact costs would wipe out the net alpha. This tells the committee how much capital the strategy can absorb.

_Interpretation: to be written once results are in._

---
# Part 14 : Robustness Checks

## 14.1 : Transaction Cost Sensitivity

Net Sharpe ratios and alphas of all rows under the fixed-cost scenarios (10, 25 and 50 bp) and under the lecture 3 cost model with a larger assumed fund size.

## 14.2 : NBER Recessions

Performance of each row during NBER recessions versus expansions.

## 14.3 : Appendix Tests (If Time Permits)

- Beta-hedged variant of row D: hedges the net long market exposure created when $k_t < 1$ with a short market position. If the benefit holds, it does not come from market timing.
- Equal-weighted strategy instead of value-weighted
- Quintiles instead of deciles
- Moreira-Muir with full-sample $c$ (look-ahead, for reference)
- Sub-periods by decade
- Concentration (largest position weights)

_Interpretation: to be written once results are in._

---
# Part 15 : Results Sheet & Conclusion

## 15.1 : Results Sheet Export

All headline numbers (sample dates, Sharpe ratios, drawdowns, alphas and t-stats, turnover, trigger counts, capacity) are exported to a single results sheet. The report and the slides both cite this sheet; the slides may reframe but never add numbers that are not in the report. Code, data and results are marked as the final version at submission and never re-run on new data.

## 15.2 : Evidence Summary

> **Can a momentum strategy we design keep the momentum premium, reduce its crashes, and still earn alpha after costs, out of sample?**

| Hypothesis | In-sample | Out-of-sample | Verdict |
|------------|-----------|---------------|---------|
| H1 : selected signal ≥ 12-1 | | | |
| H2 : lower drawdown and worst month, Sharpe not lower, not market timing | | | |
| H3 : positive FF5 + UMD alpha after costs | | | |

In-sample conclusions use full statistical tests with $t > 3$. Out-of-sample results are point estimates with confidence intervals, phrased as "consistent / not consistent with the in-sample result".

## 15.3 : Conclusion & Limitations

Final answer to the research question, and what the strategy adds to the firm's existing portfolio. If H3 fails (alpha against UMD not significant), the message holds as long as H2 holds: same premium, much smaller crashes.

Limitations to state: borrowing fees not modelled, assumptions behind the price impact model and fund size, limited out-of-sample power, US-only test of the echo debate, and the total number of variants tested (for transparency on data mining).